# 玉藻前文章配图生成 - 卡通风格版
使用免费 T4 GPU 生成 18 张 16:9 卡通风格图片

## 使用说明
1. 依次运行下面的单元格
2. 卡通风格脸部更自然，无写实畸形问题
3. 测试满意后生成全部 18 张

In [ ]:
# 第一步：安装依赖
!pip install -q diffusers transformers accelerate safetensors torch
print('依赖安装完成')

In [ ]:
# 第二步：检查 GPU
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# 第三步：加载模型
from diffusers import StableDiffusionPipeline
import torch

print('正在加载模型...')
pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('模型加载完成！')

In [ ]:
# 第四步：定义卡通风格提示词
cartoon_prefix = '(masterpiece, best quality:1.2), (anime style:1.3), (flat color:1.1), (clean line art:1.2), (simple beautiful face:1.3), (big expressive eyes:1.2), '

negative_prompt = 'realistic, photorealistic, 3d render, low quality, blurry, distorted, ugly, bad anatomy, watermark, signature, text, deformed face, bad face, missing eyes, extra digits, missing fingers'

prompts = [
    ('1.1_印度华阳天', cartoon_prefix + 'Ancient Indian palace, beautiful fox spirit disguised as court lady, golden robes, nine tails visible in shadow, Indian architecture, mysterious atmosphere, anime style'),
    ('1.2_中国妲己', cartoon_prefix + 'Chinese Shang dynasty palace, stunning concubine Daji, silk robes, nine fox tails shadow, bronze vessels, oracle bones, Chinese painting anime style'),
    ('1.3_日本玉藻前', cartoon_prefix + 'Japanese Heian period court, beautiful Tamamo-no-Mae in elegant kimono, entering palace, cherry blossoms, anime style'),
    ('2.1_入宫得宠', cartoon_prefix + 'Tamamo-no-Mae playing koto, Heian court ladies admiring her, palace interior, golden screens, traditional Japanese anime style'),
    ('2.2_天皇病重', cartoon_prefix + 'Sick Emperor Toba in bed, dark palace room, mysterious fox shadow on wall, dramatic lighting, anime style'),
    ('2.3_阴阳师怀疑', cartoon_prefix + 'Yin-Yang master Abe Yasunari meditating, seeing nine-tailed fox shadow, moonlight, Edo period anime style'),
    ('3.1_识破妖身', cartoon_prefix + 'Yin-Yang masters casting spells, nine-tailed fox spirit revealed, magical circles, Japanese mythology anime'),
    ('3.2_天皇的震惊', cartoon_prefix + 'Emperor Toba shocked, mirror reflection showing fox ears, palace room, dramatic lighting, anime style'),
    ('3.3_那须野的藏身', cartoon_prefix + 'Abandoned mansion in Nasu field, moonlight, mysterious atmosphere, Japanese countryside anime'),
    ('4.1_三浦介与上总介', cartoon_prefix + 'Japanese samurai warriors preparing for battle, traditional armor, Nasu field, anime style'),
    ('4.2_激战那须野', cartoon_prefix + 'Epic battle, nine-tailed fox giant form, samurai fighting, dynamic action, Japanese mythology anime'),
    ('4.3_妖狐之死', cartoon_prefix + 'Nine-tailed fox falling, arrow in forehead, dramatic sunset, tragic scene anime'),
    ('5.1_石头诞生', cartoon_prefix + 'Giant stone formation, poisonous aura, dead vegetation, Japanese landscape anime'),
    ('5.2_镇魂与封印', cartoon_prefix + 'Buddhist monk chanting sutra, glowing stone, peaceful atmosphere, Japanese temple anime'),
    ('5.3_现代遗迹', cartoon_prefix + 'Tourists visiting Sessho-seki stone, modern Japan, historical site anime'),
    ('6.1_文学与戏剧', cartoon_prefix + 'Traditional Japanese theater, Noh mask, scroll paintings, Tamamo-no-Mae story anime'),
    ('6.2_现代流行文化', cartoon_prefix + 'Anime style Tamamo-no-Mae, modern illustration, vibrant colors, manga aesthetic'),
    ('6.3_东亚妖狐文化的交融', cartoon_prefix + 'Three women India China Japan, fox spirits, cultural exchange, artistic composition anime'),
]

print(f'准备生成 {len(prompts)} 张卡通风格图片')

In [ ]:
# 测试用：只生成第一张，快速验证卡通风格脸部效果
print('=== 测试生成第一张（卡通风格）：1.1_印度华阳天 ===')
test_name, test_prompt = prompts[0]
print(f'提示词: {test_prompt}')

try:
    test_image = pipe(
        prompt=test_prompt,
        negative_prompt=negative_prompt,
        width=768,
        height=432,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]
    
    # 显示测试图
    display(test_image.resize((960, 540)))
    print('卡通风格测试图生成完成！检查脸部是否自然，如果满意，再运行下一个单元格生成全部18张')
    
except Exception as e:
    print(f'测试失败: {e}')

In [ ]:
# 第五步：生成全部图片（卡通风格）
import os
from PIL import Image

# 创建输出目录
os.makedirs('tamamo_images', exist_ok=True)

print('开始生成卡通风格图片（768x432 → 1920x1080）...\n')
for i, (name, prompt) in enumerate(prompts, 1):
    print(f'[{i}/{len(prompts)}] 生成: {name}')
    
    try:
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=768,
            height=432,
            num_inference_steps=30,
            guidance_scale=7.5,
        ).images[0]
        
        # 放大到 1920x1080
        image_large = image.resize((1920, 1080), Image.LANCZOS)
        
        # 保存
        output_path = f'tamamo_images/{name}.png'
        image_large.save(output_path, 'PNG', quality=95)
        print(f'   ✓ 保存: {output_path}')
        
    except Exception as e:
        print(f'   ✗ 失败: {e}')

print('\n所有图片生成完成！')

In [ ]:
# 第六步：打包并下载
import shutil
from google.colab import files

# 打包成 zip
shutil.make_archive('tamamo_images', 'zip', 'tamamo_images')
print('打包完成: tamamo_images.zip')

# 下载
files.download('tamamo_images.zip')